"""
# NVIDIA LipSync - Python Notebook

This notebook demonstrates how to use the NVIDIA Maxine LipSync service with Python. NVIDIA LipSync is an AI-powered service that synchronizes lip movements in videos with input speech, creating naturally synchronized speech animations.

## Overview

The service processes MP4 video files and audio inputs (WAV or MP3) to output lip-synced videos:

- **Service Integration:** Connect to Maxine LipSync services
- **Audio-Video Sync:** Synchronize lip movements with audio speech
- **Flexible Configuration:** Customize encoding, extension, and processing parameters
- **Speaker Data:** Provide per-frame speaker bounding boxes via JSON
- **Background Audio:** Mix background audio into the output
- **Streaming Support:** Process videos with real-time progress tracking

## Requirements

- **Input Video:** MP4 files with H.264 video codec (audio optional), videos with Variable Frame Rate (VFR) are not supported
- **Input Audio:** WAV or MP3 audio files for lip synchronization
- **Speaker Data:** Optional JSON file with per-frame speaker bounding boxes
- **Output:** MP4 files with H.264 video codec
- **Service:** Access to a running Maxine LipSync service instance
"""


## Installation

### Requirements
- Python 3.12. 
- pip package manager.
- gRPC dependencies from the requirements.txt file.

bash `pip install -r ../requirements.txt`

In [ ]:
%pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your Maxine LipSync service. The service can be running on your machine or on a remote server accessible from your environment.
To run the Lipsync NIM, follow the instructions in [Getting Started](https://docs.nvidia.com/nim/maxine/lipsync/latest/getting-started.html). 


In [ ]:
import os
import sys
import pathlib

# Setup paths for LipSync modules
SCRIPT_PATH = str(pathlib.Path().resolve())
print(SCRIPT_PATH)
sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../"))

# If an NVIDIA package (e.g. CUDA libs) is already installed, its __init__.py
# claims the nvidia namespace and blocks discovery of our local protobuf stubs.
# Extend its __path__ so that interfaces/nvidia/ is also searched.
try:
    import nvidia
    nvidia.__path__.append(os.path.join(SCRIPT_PATH, "../interfaces/nvidia"))
except ModuleNotFoundError:
    pass

# Service connection configuration
SERVICE_HOST = "localhost"
SERVICE_PORT = 8001
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print(f"Python paths configured for LipSync modules")

In [ ]:
import grpc
import json
import time
from typing import Iterator
from contextlib import nullcontext

# Import LipSync modules
from config import LipSyncConfig
from constants import *
import nvidia.ai4m.lipsync.v1.lipsync_pb2 as lipsync_pb2
import nvidia.ai4m.lipsync.v1.lipsync_pb2_grpc as lipsync_pb2_grpc
import nvidia.ai4m.video.v1.video_pb2 as video_pb2
import nvidia.ai4m.audio.v1.audio_pb2 as audio_pb2
import nvidia.ai4m.common.v1.common_pb2 as common_pb2
from utils.utils import create_channel_credentials, create_protobuf_any_value

print("All libraries imported successfully!")

## Helper Functions

Define functions for processing video, audio, and speaker data and communicating with the LipSync service. These functions handle the bi-directional gRPC streaming protocol used by the LipSync service.

### Function Overview

The LipSync service uses bi-directional gRPC streaming over a secure channel. The implementation consists of two main functions:

1. **Request Generator:** A Python iterator that yields request chunks to stream to the service
2. **Response Processor:** A function that processes the incoming gRPC data stream and writes the output file

### Streaming Protocol Details

- **First Stream Item:** Configuration object that sets the LipSync feature parameters
- **Subsequent Items:** Interleaved chunks of video data, audio data, speaker data, and background audio (64KB each)
- **Response Stream:** Video data chunks containing the processed output MP4 file
- **Configuration Echo:** If parameters are sent, the first response item is an echo that should be skipped


In [ ]:
def create_custom_encoding_params(params: dict) -> video_pb2.CustomEncodingParams:
    """Create CustomEncodingParams from a dictionary.

    Args:
        params: Dictionary with string keys and values of type bool, int, float, or str

    Returns:
        CustomEncodingParams protobuf message
    """
    custom_params = video_pb2.CustomEncodingParams()
    for key, value in params.items():
        custom_params.custom[key].CopyFrom(create_protobuf_any_value(value))
    return custom_params


def batched_json_reader(frames: list, batch_size: int) -> Iterator[list]:
    """Read JSON frame data in batches.

    Args:
        frames: List of frame dictionaries from JSON
        batch_size: Number of frames to include in each batch

    Yields:
        List of frames in batches of the specified size
    """
    for i in range(0, len(frames), batch_size):
        yield frames[i : i + batch_size]


def validate_input_files(
    video_filepath: str,
    audio_filepath: str,
    speaker_data_input: str | None,
    background_audio_path: str | None,
) -> None:
    """Validate that all provided input file paths exist.

    Args:
        video_filepath: Path to the input video file
        audio_filepath: Path to the input audio file
        speaker_data_input: Optional path to speaker data JSON file
        background_audio_path: Optional path to background audio file

    Raises:
        FileNotFoundError: If any provided file path does not exist
    """
    if not os.path.exists(video_filepath):
        raise FileNotFoundError(f"Video file not found: {video_filepath}")
    if not os.path.exists(audio_filepath):
        raise FileNotFoundError(f"Audio file not found: {audio_filepath}")
    if speaker_data_input and not os.path.exists(speaker_data_input):
        raise FileNotFoundError(f"Speaker data file not found: {speaker_data_input}")
    if background_audio_path and not os.path.exists(background_audio_path):
        raise FileNotFoundError(f"Background audio file not found: {background_audio_path}")


def build_background_audio_config(
    background_audio_path: str | None,
    background_audio_volume: float,
) -> lipsync_pb2.BackgroundAudioConfig | None:
    """Build a BackgroundAudioConfig protobuf from a file path and volume.

    Args:
        background_audio_path: Path to background audio file, or None
        background_audio_volume: Volume level for background audio

    Returns:
        BackgroundAudioConfig message, or None if no path provided
    """
    if not background_audio_path:
        return None
    bg_ext = os.path.splitext(background_audio_path)[1].lower().lstrip(".")
    return lipsync_pb2.BackgroundAudioConfig(
        is_background_audio_provided=True,
        audio_codec=AUDIO_CODEC_CONFIGS.get(bg_ext, audio_pb2.AudioCodec.AUDIO_CODEC_WAV),
        audio_volume=background_audio_volume,
    )


def load_speaker_data(speaker_data_input: str | None) -> list | None:
    """Load per-frame speaker data from a JSON file.

    Args:
        speaker_data_input: Path to the JSON file, or None

    Returns:
        List of frame dictionaries from the "frames" key, or None if no path provided
    """
    if not speaker_data_input:
        return None
    print(f"Loading JSON speaker data from {speaker_data_input}")
    with open(speaker_data_input, "r") as json_fd:
        json_data = json.load(json_fd)
        frames = json_data.get("frames", [])
    print(f"Loaded {len(frames)} frames from JSON")
    return frames


def parse_speaker_frame(
    frame_data: dict, frame_index: int
) -> lipsync_pb2.SpeakerInfoPerFrame:
    """Parse a single JSON frame dict into a SpeakerInfoPerFrame protobuf.

    Handles the default case (no speakers detected) and the multi-speaker case
    with bounding box, speaker_id, and is_speaking fields.

    Args:
        frame_data: Dictionary for one frame from the speaker data JSON
        frame_index: Global frame index to assign as frame_id

    Returns:
        SpeakerInfoPerFrame protobuf message
    """
    speaker_data_results = frame_data.get("speakers", [])

    bypass_kwargs = {}
    if "bypass" in frame_data:
        bypass_kwargs["bypass"] = bool(frame_data["bypass"])

    if not speaker_data_results:
        return lipsync_pb2.SpeakerInfoPerFrame(
            frame_id=frame_index,
            speaker_infos=[
                lipsync_pb2.SpeakerInfo(
                    speaker_bbox=common_pb2.BoundingBox(x=0, y=0, width=0, height=0),
                    speaker_id=0,
                    is_speaking=True,
                )
            ],
            **bypass_kwargs,
        )

    speakers = []
    for speaker_data in speaker_data_results:
        bbox = speaker_data.get("bbox", [0, 0, 0, 0])
        x, y, width, height = bbox

        speaker_id = speaker_data.get("speaker_id")
        if speaker_id is None:
            speaker_id = speaker_data.get("face_tracker_id", 0)

        is_speaking = speaker_data.get("is_speaking", False)

        speakers.append(
            lipsync_pb2.SpeakerInfo(
                speaker_bbox=common_pb2.BoundingBox(
                    x=float(x), y=float(y), width=float(width), height=float(height),
                ),
                speaker_id=int(speaker_id),
                is_speaking=bool(is_speaking),
            )
        )

    return lipsync_pb2.SpeakerInfoPerFrame(
        frame_id=frame_index,
        speaker_infos=speakers,
        **bypass_kwargs,
    )


def generate_request_for_inference(
    video_filepath: str,
    audio_filepath: str,
    config_params: dict | None = None,
    speaker_data_input: str | None = None,
    background_audio_path: str | None = None,
    background_audio_volume: float = 0.5,
) -> Iterator[lipsync_pb2.LipsyncRequest]:
    """Generate streaming requests for the LipSync service.
    
    This function creates a Python iterator that yields gRPC request messages for streaming
    to the LipSync service. The first yielded item contains configuration parameters
    (if provided), followed by interleaved chunks of video, audio, speaker data, and
    background audio.
    
    Args:
        video_filepath: Path to the input MP4 video file (H.264 codec recommended)
        audio_filepath: Path to the input audio file (WAV or MP3)
        config_params: Dictionary of LipSync configuration parameters. If None or empty,
                      default values will be used by the service.
        speaker_data_input: Optional path to JSON file containing per-frame speaker data
        background_audio_path: Optional path to background audio file to mix into output
        background_audio_volume: Volume level for background audio (default 1.0)

    Yields:
        LipsyncRequest messages containing either of the following:
        - Configuration object (first yield, only if config_params provided)
        - Video, audio, speaker data, or background audio data chunks
        
    Raises:
        IOError: If input files cannot be read due to permissions or I/O errors
        FileNotFoundError: If input files don't exist at the specified paths
    """

    validate_input_files(video_filepath, audio_filepath, speaker_data_input, background_audio_path)

    # Add background audio config to params if needed
    bg_audio_config = build_background_audio_config(background_audio_path, background_audio_volume)
    if bg_audio_config and config_params:
        config_params["background_audio_config"] = bg_audio_config

    # Send configuration first
    if config_params:
        print("Sending configuration parameters")
        yield lipsync_pb2.LipsyncRequest(
            config=lipsync_pb2.LipsyncConfig(**config_params)
        )
    
    # Load JSON speaker data if provided
    json_frames = None
    if speaker_data_input and config_params and config_params.get("is_speaker_info_provided", False):
        json_frames = load_speaker_data(speaker_data_input)

    # Send video, audio, speaker data, and background audio in interleaved chunks with progress tracking
    print("Sending input data")
    
    video_done = False
    audio_done = False
    speaker_done = True if not json_frames else False
    bg_audio_done = True if not background_audio_path else False
    
    try:
        with open(video_filepath, "rb") as video_file, open(
            audio_filepath, "rb"
        ) as audio_file, (
            open(background_audio_path, "rb") if background_audio_path else nullcontext()
        ) as bg_audio_file:

            speaker_data_batch_iter = None
            global_frame_index = 0
            if json_frames:
                speaker_data_batch_iter = batched_json_reader(json_frames, SPEAKER_DATA_BATCH_SIZE)

            video_chunk_count = 0
            audio_chunk_count = 0
            speaker_batch_count = 0
            
            while not (video_done and audio_done and speaker_done and bg_audio_done):
                # Send video chunk if not done
                if not video_done:
                    try:
                        video_buffer = video_file.read(DATA_CHUNK_SIZE)
                        if video_buffer == b"":
                            video_done = True
                        else:
                            video_chunk_count += 1
                            yield lipsync_pb2.LipsyncRequest(
                                input=lipsync_pb2.LipsyncInputData(video_file_data=video_buffer)
                            )
                    except IOError as e:
                        print(f"Error reading video chunk {video_chunk_count}: {e}")
                        raise

                # Send audio chunk if not done
                if not audio_done:
                    try:
                        audio_buffer = audio_file.read(DATA_CHUNK_SIZE)
                        if audio_buffer == b"":
                            audio_done = True
                        else:
                            audio_chunk_count += 1
                            yield lipsync_pb2.LipsyncRequest(
                                input=lipsync_pb2.LipsyncInputData(audio_file_data=audio_buffer)
                            )
                    except IOError as e:
                        print(f"Error reading audio chunk {audio_chunk_count}: {e}")
                        raise

                # Send speaker data batch if not done
                if not speaker_done and speaker_data_batch_iter:
                    try:
                        batch = next(speaker_data_batch_iter)
                        speaker_batch_count += 1

                        speaker_info_per_frame_batch = [
                            parse_speaker_frame(frame_data, global_frame_index + batch_idx)
                            for batch_idx, frame_data in enumerate(batch)
                        ]

                        global_frame_index += len(batch)

                        yield lipsync_pb2.LipsyncRequest(
                            input=lipsync_pb2.LipsyncInputData(
                                per_frame_speaker_infos=speaker_info_per_frame_batch
                            )
                        )
                    except StopIteration:
                        speaker_done = True
                    except Exception as e:
                        print(f"Error processing speaker data batch {speaker_batch_count}: {e}")
                        raise

                # Send background audio chunk if not done
                if not bg_audio_done:
                    try:
                        bg_buffer = bg_audio_file.read(DATA_CHUNK_SIZE)
                        if bg_buffer == b"":
                            bg_audio_done = True
                        else:
                            yield lipsync_pb2.LipsyncRequest(
                                input=lipsync_pb2.LipsyncInputData(
                                    background_audio_file_data=bg_buffer
                                )
                            )
                    except IOError as e:
                        print(f"Error reading background audio: {e}")
                        raise

            print(f"Upload complete: {video_chunk_count} video chunks, {audio_chunk_count} audio chunks")
            if speaker_batch_count > 0:
                print(f"Speaker data batches sent: {speaker_batch_count}")
                    
    except IOError as e:
        print(f"Error reading input files: {e}")
        raise

In [5]:
def write_output_file_from_response(
    response_iter: Iterator[lipsync_pb2.LipsyncResponse],
    output_filepath: os.PathLike = "output.mp4",
) -> None:
    """Function to write the output file from the incoming gRPC data stream.

    Args:
        response_iter: Responses from the server to write into output file
        output_filepath: Path to output file
    """
    print(f"Writing output to {output_filepath}")

    chunk_count = 0
    total_bytes = 0
    last_mb_reported = 0

    with open(output_filepath, "wb") as fd:
        for response in response_iter:
            if response.HasField("video_file_data"):
                chunk_data = response.video_file_data
                fd.write(chunk_data)

                chunk_count += 1
                total_bytes += len(chunk_data)

                mb_received = total_bytes // (1024 * 1024)
                if mb_received >= last_mb_reported + 10:
                    last_mb_reported = mb_received
                    print(f"  Receiving: {mb_received} MB received ({chunk_count} chunks)", flush=True)

    print(f"Download complete: {chunk_count} chunks ({total_bytes / (1024*1024):.1f} MB total)")

## Video Processing 

Process video and audio files with the LipSync service. This method runs the LipSync effect.


In [ ]:
# Process the video
def process_lipsync_video(
    video_path: str, 
    audio_path: str, 
    output_path: str, 
    config: dict,
    speaker_data_filepath: str | None = None,
    background_audio_path: str | None = None,
    background_audio_volume: float = 1.0,
) -> bool:
    """Process video with LipSync service.
    
    Args:
        video_path: Path to input video file
        audio_path: Path to input audio file
        output_path: Path for output video file
        config: LipSync configuration dictionary
        speaker_data_filepath: Optional path to JSON file with per-frame speaker data
        background_audio_path: Optional path to background audio file to mix into output
        background_audio_volume: Volume level for background audio (default 1.0)
    
    Returns:
        True if processing succeeded
    """
    try:
        print(f"\nProcessing video: {video_path}")
        print(f"Processing audio: {audio_path}")
        if speaker_data_filepath:
            print(f"Using speaker data: {speaker_data_filepath}")
        if background_audio_path:
            print(f"Using background audio: {background_audio_path} (volume: {background_audio_volume})")
        print(f"Connecting to service: {SERVICE_TARGET}")
        
        # Connect to service (use secure channel if needed)
        with grpc.insecure_channel(SERVICE_TARGET) as channel:
            stub = lipsync_pb2_grpc.LipSyncServiceStub(channel)
            
            start_time = time.time()
            
            # Process video
            responses = stub.Lipsync(
                generate_request_for_inference(
                    video_path, audio_path, config,
                    speaker_data_filepath=speaker_data_filepath,
                    background_audio_path=background_audio_path,
                    background_audio_volume=background_audio_volume,
                )
            )
            
            # Skip configuration echo response
            next(responses)
            
            # Write output with progress tracking
            write_output_file_from_response(responses, output_path)
            
            end_time = time.time()
            processing_time = end_time - start_time
            
            print(f"Processing complete in {processing_time:.1f}s")
            print(f"Output saved: {output_path}")
            
            return True
            
    except FileNotFoundError as e:
        print(f"Input file not found: {e}")
        print("   Please update file paths with valid video and audio files")
        return False
    except grpc.RpcError as e:
        print(f"Service connection failed: {e}")
        print(f"   Ensure the LipSync service is running at {SERVICE_TARGET}")
        return False
    except Exception as e:
        print(f"Processing failed: {e}")
        return False

## Run LipSync Processing - Example

This cell demonstrates how to run LipSync processing on an example video (.mp4) and speech input (.wav) file. The configuration uses lossy encoding with a video bitrate set to 30 Mbps for the output video.

In [ ]:
# Configuration
video_filepath = "../assets/sample_video_streamable.mp4"         # Update with your input video path
audio_filepath = "../assets/sample_audio.wav"                    # Update with your input audio path  
output_filepath = "lipsync_output.mp4"                              # Desired output video path

def create_lipsync_config() -> dict:
    """Create LipSync configuration with default parameters.
    
    Uses the standard default values from the LipSync service configuration.
    You can modify these values as needed for your use case.
    
    Returns:
        Configuration dictionary for the LipSync service
    """
    
    # Default configuration using correct string values from README
    config = {
        "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"],           # Audio codec handling
        "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],        # Audio extension behavior
        "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],        # Video extension behavior
        "is_speaker_info_provided": False,                          # No speaker data provided
        "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],          # Output audio codec
        "output_video_encoding": video_pb2.VideoEncoding(
            lossy=video_pb2.LossyEncoding(
                bitrate_mbps=30,              # Video bitrate (30 Mbps default)
                idr_interval=8                # IDR frame interval (8 default)
            )
        )
    }
    
    return config

config_params = create_lipsync_config()  # Standard lossy encoding

# Execute processing
success = process_lipsync_video(
    video_filepath, 
    audio_filepath, 
    output_filepath, 
    config_params,
)


## Advanced Usage Examples

Examples demonstrating different parameter combinations and use cases.

#### Encoding Options

##### Standard Lossy Encoding
```python
video_pb2.VideoEncoding(
    lossy=video_pb2.LossyEncoding(
        bitrate_mbps=30,      # Video bitrate in Mbps (default: 30)
        idr_interval=8        # IDR frame interval (default: 8)
    )
)
```

##### Lossless Encoding
```python
video_pb2.VideoEncoding(lossless=True)
```
Enables lossless video encoding. This setting overrides any bitrate configuration to ensure maximum quality output, although it results in larger file sizes.


##### Custom Encoding Parameters
```python
video_pb2.VideoEncoding(
    custom_encoding=create_custom_encoding_params({
        "idrinterval": 20,
        "maxbitrate": 3000000,
    })
)
```

**Note:** <span style="color:red">Custom encoding parameters are for expert users who need fine-grained control over video encoding. Incorrect values can cause encoding failures or poor-quality output. To configure the nvenc encoder, refer to [Gst properties of the Gst-nvvideo4linux2 encoder plugin](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_gst-nvvideo4linux2.html#:~:text=The%20following%20table%20summarizes%20the%20Gst%20properties%20of%20the%20Gst%2Dnvvideo4linux2%20encoder%20plugin).</span>

In [8]:
# Encoding Configuration Examples

def create_high_bitrate_config() -> dict:
    """Create high-bitrate LipSync configuration."""
    return {
        "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
        "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
        "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
        "is_speaker_info_provided": False, 
        "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
        "output_video_encoding": video_pb2.VideoEncoding(
            lossy=video_pb2.LossyEncoding(
                bitrate_mbps=40,         # Higher bitrate for better quality
                idr_interval=4           # More frequent IDR frames
            )
        )
    }


def create_lossless_config() -> dict:
    """Create configuration for archival/preservation quality."""
    return {
        "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
        "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"], 
        "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"], 
        "is_speaker_info_provided": False, 
        "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
        "output_video_encoding": video_pb2.VideoEncoding(lossless=True) # Lossless quality
    }

def create_low_bitrate_config() -> dict:
    """Create configuration optimized for streaming/web delivery."""
    return {
        "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"],        
        "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"], 
        "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
        "is_speaker_info_provided": False, 
        "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
        "output_video_encoding": video_pb2.VideoEncoding(
            lossy=video_pb2.LossyEncoding(
                bitrate_mbps=3,  # Lower bitrate for lower quality
                idr_interval=6
            )
        )
    }

configs = {
        "standard": create_lipsync_config(),
        "high_bitrate": create_high_bitrate_config(),
        "low_bitrate": create_low_bitrate_config(),
        "lossless": create_lossless_config(),
    }

In [ ]:
# Process with standard configuration
print("Processing with standard config:")
print(f"  Config: {configs['standard']}")
success = process_lipsync_video(video_filepath, audio_filepath, "output_standard.mp4", configs["standard"])

In [ ]:
# Process with lossless configuration
print("Processing with lossless config:")
print(f"  Config: {configs['lossless']}")
success = process_lipsync_video(video_filepath, audio_filepath, "output_lossless.mp4", configs["lossless"])

In [ ]:
# Process with high bitrate configuration
print("Processing with high bitrate config:")
print(f"  Config: {configs['high_bitrate']}")
success = process_lipsync_video(video_filepath, audio_filepath, "output_hq.mp4", configs["high_bitrate"])

In [ ]:

# Process with low bitrate configuration
print("Processing with low bitrate config:")
print(f"  Config: {configs['low_bitrate']}")
success = process_lipsync_video(video_filepath, audio_filepath, "output_lq.mp4", configs["low_bitrate"])

In [ ]:
def create_custom_encoding_config(custom_params: dict) -> dict:
    """Create LipSync configuration with custom encoding parameters."""
    return {
        "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
        "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
        "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
        "is_speaker_info_provided": False, 
        "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
        "output_video_encoding": video_pb2.VideoEncoding(
            custom_encoding=create_custom_encoding_params(custom_params)
        )
    }


# Example: Custom encoding parameters
def demo_custom_encoding():
    """Demonstrate custom encoding parameter usage."""
    
    # Expert-level custom parameters
    # Refer to [deepstream encoder properties](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_gst-nvvideo4linux2.html#:~:text=The%20following%20table%20summarizes%20the%20Gst%20properties%20of%20the%20Gst%2Dnvvideo4linux2%20encoder%20plugin
    custom_params = {
        "idrinterval": 20,           # IDR frame interval
        "maxbitrate": 4000000,       # Maximum bitrate
    }

    
    config = create_custom_encoding_config(custom_params)
    
    print("Custom encoding parameters:")
    for key, value in custom_params.items():
        print(f"  {key}: {value}")
    
    print("\nNote: Custom parameters require expert knowledge of GStreamer nvvideo4linux2 encoder")
    
    success = process_lipsync_video(video_filepath, audio_filepath, "output_custom.mp4", config)

demo_custom_encoding()

#### Speaker data

The speaker data JSON file provides per-frame bounding box and speaker metadata so that the LipSync NIM can target specific facial regions rather than relying on automatic face detection.

- **Per-frame control**: Each frame can independently be set to bypass or process
- **Speaker data required**: Set `is_speaker_info_provided: True` in the config and provide a JSON speaker data file
- **JSON format**: Each frame entry in the JSON can include a `"bypass": true` field

Example speaker data JSON with bypass:
```json
{
    "frames": [
        {"speakers": [{"bbox": [0, 0, 0, 0], "speaker_id": 0, "is_speaking": true}], "bypass": true},
        {"speakers": [{"bbox": [100, 50, 200, 200], "speaker_id": 0, "is_speaking": true}], "bypass": false},
        ...
    ]
}
```

In [ ]:
video_filepath = "../assets/sample_video_streamable.mp4"         # Update with your input video path
audio_filepath = "../assets/sample_audio.wav"                    # Update with your input audio path
speaker_data_filepath = "../assets/sample_speaker_data.json"    # Update with your speaker data JSON path
output_filepath = "lipsync_bypass_output.mp4"                    # Desired output video path

# Configuration with speaker info provided for bypass demo
config_with_speaker_info = {
    "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"],
    "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
    "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
    "is_speaker_info_provided": True,
    "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
    "output_video_encoding": video_pb2.VideoEncoding(
        lossy=video_pb2.LossyEncoding(
            bitrate_mbps=30,
            idr_interval=8
        )
    )
}

# Process with speaker data (which may contain per-frame bypass flags)
success = process_lipsync_video(
    video_filepath, audio_filepath, output_filepath,
    config_with_speaker_info,
    speaker_data_filepath=speaker_data_filepath,
)

#### Bypass LipSync for a Percentage of Frames

This example demonstrates how to bypass LipSync processing for the last portion of a video. You can configure what percentage of total frames should be bypassed — for instance, setting `bypass_percentage = 50` will apply LipSync to the first half of the video and pass the second half through unchanged.

This is useful when you only want to lip-sync a portion of a video (e.g., the speaker talks in the first half, and the second half is B-roll or a different scene).

For more granular control for bypass, please set bypass per frame in the speaker-data-input json file for all the frames you want to bypass Lipsync for.

In [12]:
import copy

def create_bypass_speaker_data(speaker_data_path: str, bypass_percentage: float) -> list:
    """Load speaker data and set bypass=True for the last N% of frames.

    Args:
        speaker_data_path: Path to the source speaker data JSON file.
        bypass_percentage: Percentage of total frames (from the end) to bypass (0-100).

    Returns:
        Modified list of frame dictionaries with bypass flags set.
    """
    if not 0 <= bypass_percentage <= 100:
        raise ValueError("bypass_percentage must be between 0 and 100")

    with open(speaker_data_path, "r") as f:
        data = json.load(f)

    frames = copy.deepcopy(data.get("frames", []))
    total_frames = len(frames)
    bypass_start = total_frames - int(total_frames * bypass_percentage / 100)

    for i, frame in enumerate(frames):
        frame["bypass"] = i >= bypass_start

    bypassed = sum(1 for f in frames if f["bypass"])
    print(f"Total frames: {total_frames}")
    print(f"LipSync applied: frames 0-{bypass_start - 1} ({bypass_start} frames)")
    print(f"Bypass enabled:  frames {bypass_start}-{total_frames - 1} ({bypassed} frames)")

    return frames

In [ ]:
# Configure bypass percentage (0-100)
bypass_percentage = 50  # Bypass the last 50% of frames

video_filepath = "../assets/sample_video_streamable.mp4"
audio_filepath = "../assets/sample_audio.wav"
speaker_data_path = "../assets/sample_speaker_data.json"
output_filepath = f"lipsync_bypass_{bypass_percentage}pct_output.mp4"

# Generate modified speaker data with bypass flags
modified_frames = create_bypass_speaker_data(speaker_data_path, bypass_percentage)

# Write the modified speaker data to a temporary file
modified_speaker_data_path = "speaker_data_with_bypass.json"
with open(modified_speaker_data_path, "w") as f:
    json.dump({"frames": modified_frames}, f)
print(f"\nModified speaker data written to {modified_speaker_data_path}")

# Configuration with speaker info enabled
config_with_bypass = {
    "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"],
    "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
    "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
    "is_speaker_info_provided": True,
    "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
    "output_video_encoding": video_pb2.VideoEncoding(
        lossy=video_pb2.LossyEncoding(
            bitrate_mbps=30,
            idr_interval=8
        )
    )
}

# Process the video with partial bypass
success = process_lipsync_video(
    video_filepath, audio_filepath, output_filepath,
    config_with_bypass,
    speaker_data_filepath=modified_speaker_data_path,
)

#### Video and Audio Extension

- `extend_audio`: Controls how audio extension is handled when audio is shorter than video
  - `unspecified`: No audio extension is performed (default)
  - `silence`: Extend with silence to match video duration

- `extend_video`: Controls how video extension is handled when video is shorter than audio
  - `unspecified`: No video extension is performed (default)
  - `forward`: Play last 5 seconds of video frames forward repeatedly to match audio duration
  - `reverse`: Play last 5 seconds of video frames in reverse repeatedly to match audio duration

In [16]:
# Extension Configuration Examples with Execution

# Setup file paths for extension examples
video_filepath = "../assets/sample_video_streamable.mp4"  # 30s video
audio_filepath = "../assets/sample_audio.wav"             # 30s audio 

extension_video_path = "../assets/sample_video_streamable_20s.mp4"  # 20s video for testing video extension
extension_audio_path = "../assets/sample_audio_20s.wav"             # 20s audio for testing audio extension

# Output paths for different extension scenarios
output_extend_audio = "./extended_audio_output.mp4"
output_extend_video_forward = "./extended_video_forward_output.mp4"
output_extend_video_reverse = "./extended_video_reverse_output.mp4"
output_no_extension = "./no_extension_output.mp4"

# Example 1: No extension (default behavior)
config_no_extension = {
    "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
    "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
    "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
    "is_speaker_info_provided": False, 
    "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
    "output_video_encoding": video_pb2.VideoEncoding(
        lossy=video_pb2.LossyEncoding(
            bitrate_mbps=30,
            idr_interval=8
        )
    )
}


# Example 2: Video longer than audio - extend audio with silence
config_extend_audio = {
    "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
    "extend_audio": EXTEND_AUDIO_CONFIGS["silence"],
    "extend_video": EXTEND_VIDEO_CONFIGS["unspecified"],
    "is_speaker_info_provided": False, 
    "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
    "output_video_encoding": video_pb2.VideoEncoding(
        lossy=video_pb2.LossyEncoding(
            bitrate_mbps=30,
            idr_interval=8
        )
    )
}

# Example 3: Audio longer than video - extend video by playing last 5 seconds forward
config_extend_video_forward = {
    "input_audio_codec": AUDIO_CODEC_CONFIGS["wav"], 
    "extend_audio": EXTEND_AUDIO_CONFIGS["unspecified"],
    "extend_video": EXTEND_VIDEO_CONFIGS["forward"],
    "is_speaker_info_provided": False, 
    "output_audio_codec": AUDIO_CODEC_CONFIGS["opus"],
    "output_video_encoding": video_pb2.VideoEncoding(
        lossy=video_pb2.LossyEncoding(
            bitrate_mbps=30,
            idr_interval=8
        )
    )
}

In [ ]:
# Execute extension examples
print("=== Extension Configuration Examples ===\n")

# Execute Example 1: No extension
print("1. Processing with no extension (default)...")
success4 = process_lipsync_video(
    video_filepath,
    audio_filepath,
    output_no_extension,
    config_no_extension,
)
print(f"   Result: {'Success' if success4 else 'Failed'}")
print(f"   Output: {output_no_extension}\n")

In [ ]:
# Execute Example 2: Extend audio with silence
print("2. Processing with audio extension (silence)...")
success1 = process_lipsync_video(
    video_filepath,
    extension_audio_path,
    output_extend_audio,
    config_extend_audio,
)
print(f"   Result: {'Success' if success1 else 'Failed'}")
print(f"   Output: {output_extend_audio}\n")

In [ ]:
# Execute Example 3: Extend video forward
# Note: Video extension can take much more time to process.
print("3. Processing with video extension (forward)...")
success2 = process_lipsync_video(
    extension_video_path,
    audio_filepath,
    output_extend_video_forward,
    config_extend_video_forward,
)
print(f"   Result: {'Success' if success2 else 'Failed'}")
print(f"   Output: {output_extend_video_forward}\n")